In [ ]:
import os, torch
HF_TOKEN = ""
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
from huggingface_hub import login
login(token=HF_TOKEN)

TTS_MODEL = "ARTPARK-IISc/DhVaani-0.5"
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("logged in |", DEV)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


logged in | cuda


In [3]:
!pip install -q vocos
print("deps ready")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 13.3 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
deps ready


In [4]:
from transformers import AutoModel

model = AutoModel.from_pretrained(
    TTS_MODEL, trust_remote_code=True, token=HF_TOKEN,
    low_cpu_mem_usage=False, device_map=None,
).to(DEV).eval()

for m in model.modules():                       
    pe = getattr(m, "pe", None)
    if isinstance(pe, torch.Tensor) and pe.is_meta:
        m.pe = None
print("model loaded on", DEV)

config.json:   0%|          | 0.00/882 [00:00<?, ?B/s]

modeling_dhvaani.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ARTPARK-IISc/DhVaani-0.5:
- modeling_dhvaani.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/491M [00:00<?, ?B/s]

Fetching 29 files:   0%|          | 0/29 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/889 [00:00<?, ?it/s]

model loaded on cuda


In [21]:
import librosa, soundfile as sf

REF_AUDIO   = "/kaggle/working/clip_10s_final.ogg"   # <-- your input clip
PROMPT_TEXT = ""                                               # leave "" to skip; add the clip's words for a better clone

y, _ = librosa.load(REF_AUDIO, sr=24000, mono=True)
sf.write("/kaggle/working/prompt_ref(2)_final_10.wav", y, 24000, subtype="PCM_16")
print("reference ready:", f"{len(y)/24000:.1f}s")

texts = {
    "hindi":   "मधुमेह के रोगियों को अपने आहार में मीठे और तले हुए पदार्थों से बचना चाहिए, "
               "रोज़ थोड़ा व्यायाम करना चाहिए और समय-समय पर अपने रक्त शर्करा की जाँच करवानी चाहिए।",
    "marathi": "मधुमेह असलेल्या रुग्णांनी गोड आणि तळलेले पदार्थ टाळावेत, दररोज थोडा व्यायाम करावा "
               "आणि नियमितपणे आपल्या रक्तातील साखरेची तपासणी करून घ्यावी.",
    "tamil":   "நீரிழிவு நோயாளிகள் இனிப்பு மற்றும் எண்ணெயில் பொரித்த உணவுகளைத் தவிர்க்க வேண்டும், "
               "தினமும் சிறிது உடற்பயிற்சி செய்ய வேண்டும், மேலும் அவ்வப்போது தங்கள் இரத்த சர்க்கரை அளவை பரிசோதித்துக் கொள்ள வேண்டும்.",
    "telugu":  "మధుమేహ రోగులు తీపి మరియు నూనెలో వేయించిన ఆహారాలకు దూరంగా ఉండాలి, "
               "ప్రతిరోజూ కొంత వ్యాయామం చేయాలి, మరియు అప్పుడప్పుడు తమ రక్తంలో చక్కెర స్థాయిని పరీక్షించుకోవాలి.",
}

reference ready: 10.0s


In [6]:
# !rm -rf /kaggle/working/*


In [9]:
from IPython.display import Audio, display

PROMPT_TEXT = "डायबिटीज़ के मरीज़ को खाने में किन चीज़ों से परेशानियां हो सकती हैं"   

for lang, txt in texts.items():
    audio = model.synthesize(text=txt, prompt_wav="/kaggle/working/prompt_ref.wav", prompt_text=PROMPT_TEXT)
    path = f"/kaggle/working/out_{lang}.wav"
    sf.write(path, audio, model.sampling_rate)
    print(f"\n=== {lang} ===\n{txt}\n-> {path}")
    display(Audio(path))


=== hindi ===
मधुमेह के रोगियों को अपने आहार में मीठे और तले हुए पदार्थों से बचना चाहिए, रोज़ थोड़ा व्यायाम करना चाहिए और समय-समय पर अपने रक्त शर्करा की जाँच करवानी चाहिए।
-> /kaggle/working/out_hindi.wav



=== marathi ===
मधुमेह असलेल्या रुग्णांनी गोड आणि तळलेले पदार्थ टाळावेत, दररोज थोडा व्यायाम करावा आणि नियमितपणे आपल्या रक्तातील साखरेची तपासणी करून घ्यावी.
-> /kaggle/working/out_marathi.wav



=== tamil ===
நீரிழிவு நோயாளிகள் இனிப்பு மற்றும் எண்ணெயில் பொரித்த உணவுகளைத் தவிர்க்க வேண்டும், தினமும் சிறிது உடற்பயிற்சி செய்ய வேண்டும், மேலும் அவ்வப்போது தங்கள் இரத்த சர்க்கரை அளவை பரிசோதித்துக் கொள்ள வேண்டும்.
-> /kaggle/working/out_tamil.wav



=== telugu ===
మధుమేహ రోగులు తీపి మరియు నూనెలో వేయించిన ఆహారాలకు దూరంగా ఉండాలి, ప్రతిరోజూ కొంత వ్యాయామం చేయాలి, మరియు అప్పుడప్పుడు తమ రక్తంలో చక్కెర స్థాయిని పరీక్షించుకోవాలి.
-> /kaggle/working/out_telugu.wav


In [19]:
!ffmpeg -i "/kaggle/input/datasets/henil2132/namo-audio/Narendra_Modi_voice.ogg" -t 10 -c copy "/kaggle/working/clip_10s_final.ogg"

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [24]:
import glob, os, shutil
from safetensors.torch import load_file, save_file

# 1) locate the downloaded DhVaani snapshot (complete: weights + code + _backend)
snap = sorted(glob.glob(os.path.expanduser(
    "~/.cache/huggingface/hub/models--ARTPARK-IISc--DhVaani-0.5/snapshots/*")))[-1]
print("snapshot:", snap)

# 2) copy to /kaggle/working (dereference symlinks -> self-contained folder)
DST = "/kaggle/working/DhVaani-0.5-fp16"
if os.path.exists(DST): shutil.rmtree(DST)
shutil.copytree(snap, DST, symlinks=False)

# 3) cast .safetensors weights to fp16 (shrinks fp32; no-op if already fp16)
for wf in glob.glob(f"{DST}/**/*.safetensors", recursive=True):
    sd = load_file(wf)
    sd = {k: (v.half() if v.is_floating_point() else v) for k, v in sd.items()}
    save_file(sd, wf)
    print("fp16:", os.path.relpath(wf, DST), f"{os.path.getsize(wf)/1e6:.0f} MB")

# 4) zip into /kaggle/working
zip_path = shutil.make_archive("/kaggle/working/DhVaani-0.5-fp16", "zip", DST)
print("\nZIP READY ->", zip_path, f"({os.path.getsize(zip_path)/1e6:.0f} MB)")
print("Download it from Kaggle's Output panel (right side).")

snapshot: /root/.cache/huggingface/hub/models--ARTPARK-IISc--DhVaani-0.5/snapshots/b079f01592e987042b86f4a01c3909cef569d247
fp16: model.safetensors 246 MB

ZIP READY -> /kaggle/working/DhVaani-0.5-fp16.zip (228 MB)
Download it from Kaggle's Output panel (right side).


In [25]:
from transformers import AutoModel
model = AutoModel.from_pretrained("DhVaani-0.5-fp16", trust_remote_code=True,
                                  low_cpu_mem_usage=False).half().to("cuda").eval()

Loading weights:   0%|          | 0/889 [00:00<?, ?it/s]

In [26]:
import glob, os, shutil, torch
from safetensors.torch import load_file, save_file

# 1) locate the downloaded DhVaani snapshot (weights + custom code + _backend)
snap = sorted(glob.glob(os.path.expanduser(
    "~/.cache/huggingface/hub/models--ARTPARK-IISc--DhVaani-0.5/snapshots/*")))[-1]
print("snapshot:", snap)

# 2) copy to /kaggle/working (dereference symlinks -> self-contained)
DST = "/kaggle/working/DhVaani-0.5-bf16"
if os.path.exists(DST): shutil.rmtree(DST)
shutil.copytree(snap, DST, symlinks=False)

# 3) cast .safetensors weights to bfloat16 (CPU-friendly 16-bit)
for wf in glob.glob(f"{DST}/**/*.safetensors", recursive=True):
    sd = load_file(wf)
    sd = {k: (v.bfloat16() if v.is_floating_point() else v) for k, v in sd.items()}
    save_file(sd, wf)
    print("bf16:", os.path.relpath(wf, DST), f"{os.path.getsize(wf)/1e6:.0f} MB")

# 4) zip into /kaggle/working
zip_path = shutil.make_archive("/kaggle/working/DhVaani-0.5-bf16", "zip", DST)
print("\nZIP READY ->", zip_path, f"({os.path.getsize(zip_path)/1e6:.0f} MB)")
print("Download it from Kaggle's Output panel (right side).")

snapshot: /root/.cache/huggingface/hub/models--ARTPARK-IISc--DhVaani-0.5/snapshots/b079f01592e987042b86f4a01c3909cef569d247
bf16: model.safetensors 246 MB

ZIP READY -> /kaggle/working/DhVaani-0.5-bf16.zip (196 MB)
Download it from Kaggle's Output panel (right side).
